# 01 - Ingest Genie telemetry
Writes the raw layer that `02_build_gold_tables` consumes. Both modes produce identical table shapes.

| Table | production source | synthetic |
| --- | --- | --- |
| `raw_genie_audit_events` | `system.access.audit` (`aibiGenie`, `genieChat`) | generated |
| `raw_genie_api_messages` | Genie Conversation API (question/answer text, status, timings) | n/a |
| `raw_genie_messages` | API messages + audit message events, user email from audit, feedback from audit | generated |
| `dim_genie_spaces` | Genie spaces API | generated |
| `dim_genie_users` | distinct users + optional `user_attributes_table` | generated |

**Permissions (production):** `SELECT` on `system.access.audit`; for conversation text the run-as identity needs **CAN MANAGE** on the Genie spaces (to list all users' conversations).

In [0]:
# Job parameters (set by the bundle job; defaults let you run this notebook interactively)
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "genie_churn")
dbutils.widgets.dropdown("data_mode", "production", ["production", "synthetic"])
dbutils.widgets.text("lookback_days", "180")
dbutils.widgets.dropdown("include_conversation_text", "true", ["true", "false"])
dbutils.widgets.dropdown("workspace_scope", "current", ["current", "all"])
dbutils.widgets.text("user_attributes_table", "")  # optional: catalog.schema.table with user_email, department, role
dbutils.widgets.text("max_conversations_per_space", "5000")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
DATA_MODE = dbutils.widgets.get("data_mode")
LOOKBACK_DAYS = int(dbutils.widgets.get("lookback_days"))
INCLUDE_TEXT = dbutils.widgets.get("include_conversation_text").lower() == "true"
WORKSPACE_SCOPE = dbutils.widgets.get("workspace_scope")
USER_ATTR_TABLE = dbutils.widgets.get("user_attributes_table").strip()
MAX_CONV = int(dbutils.widgets.get("max_conversations_per_space"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
print(f"Target: {CATALOG}.{SCHEMA} | mode={DATA_MODE} | lookback={LOOKBACK_DAYS}d | text={INCLUDE_TEXT} | scope={WORKSPACE_SCOPE}")

In [0]:
# Synthetic Genie telemetry for demos. Bad experiences raise churn probability, so the gold layer has a real signal.
import random, uuid
from datetime import datetime, timedelta
import pandas as pd

def generate_synthetic(n_users=450, months_back=6, seed=42):
    random.seed(seed)
    today = datetime.combine(datetime.utcnow().date(), datetime.min.time())
    start = today - timedelta(days=30 * months_back)
    spaces = [
        ("sp_sales",   "Sales Pipeline",       "Sales",        (0.05, 0.08, 0.05, 0.03)),
        ("sp_finance", "Finance P&L",          "Finance",      (0.10, 0.18, 0.10, 0.05)),
        ("sp_scm",     "Supply Chain KPIs",    "Supply Chain", (0.12, 0.10, 0.08, 0.25)),
        ("sp_ops",     "Operations Dashboard", "Operations",   (0.08, 0.07, 0.30, 0.04)),
        ("sp_hr",      "HR Workforce",         "HR",           (0.20, 0.12, 0.06, 0.10)),
    ]
    questions = {
        "sp_sales":   ["What is pipeline by region this quarter?", "Top 10 opportunities by amount", "Win rate by rep last month", "How many deals closed in Q3?"],
        "sp_finance": ["What is gross margin by business unit?", "Show opex vs budget YTD", "Revenue trend last 12 months", "EBITDA by quarter"],
        "sp_scm":     ["Which suppliers have the longest lead time?", "Late POs by category last quarter", "Inventory turns by warehouse", "Spend by supplier this year"],
        "sp_ops":     ["Daily throughput by plant", "Downtime hours by line last week", "OEE trend by site", "Work orders open over 30 days"],
        "sp_hr":      ["Headcount by department", "Attrition rate by quarter", "Open requisitions by region", "Average time to hire"],
    }
    resp_ok = "Here are the results. I queried the relevant table and aggregated by the requested dimension."
    resp_fail = ["I encountered an error executing the query: TABLE_OR_VIEW_NOT_FOUND.", "The query failed due to a timeout on the SQL warehouse.", "I was unable to generate valid SQL for this question."]
    resp_missing = ["I could not find a column that represents '{x}' in the available tables.", "The data only goes up to last year, so I cannot answer for the current period.", "There is no table containing '{x}' in this space."]
    resp_clarify = "Could you clarify which metric definition you mean? There are multiple possible columns."
    comments_neg = {
        "wrong":   ["This number doesn't match our official report.", "Wrong answer, the total is way off.", "It used the wrong date column again."],
        "slow":    ["Took forever to respond.", "Way too slow, I gave up and used Excel."],
        "missing": ["It doesn't have the data I need.", "Why is last quarter missing?", "Can't find the metric we use every day."],
        "fail":    ["Keeps erroring out.", "Another failure, useless today."],
    }
    comments_pos = ["Great, exactly what I needed!", "Super quick, thanks.", "Love this, saves me so much time."]
    depts = {"Sales": "sp_sales", "Finance": "sp_finance", "Supply Chain": "sp_scm", "Operations": "sp_ops", "HR": "sp_hr"}
    space_q = {s[0]: s[3] for s in spaces}
    users, msgs = [], []
    for i in range(n_users):
        dept = random.choices(list(depts), weights=[30, 20, 20, 20, 10])[0]
        join = start + timedelta(days=int((random.random() ** 0.7) * ((today - start).days - 3)))
        email = f"user{i:03d}@customer.com"
        users.append((email, dept, random.choice(["Analyst", "Manager", "Director", "Specialist"]), join.date()))
        sp = depts[dept] if random.random() < 0.85 else random.choice(list(space_q))
        fail, wrong, slow, missing = space_q[sp]
        freq, frustration, day = random.uniform(0.15, 0.6), 0.0, join
        while day < today:
            if day.weekday() < 5 and random.random() < freq:
                conv, prev_q = str(uuid.uuid4()), None
                for _ in range(random.randint(1, 4)):
                    ts = day + timedelta(hours=random.randint(8, 17), minutes=random.randint(0, 59))
                    q = (prev_q + " (rephrased)") if (prev_q and random.random() < 0.3) else random.choice(questions[sp])
                    r, secs = random.random(), random.uniform(4, 20)
                    if r < fail: status, outcome, resp = "FAILED", "fail", random.choice(resp_fail)
                    elif r < fail + missing: status, outcome, resp = "COMPLETED", "missing", random.choice(resp_missing).format(x=q.split()[2])
                    elif r < fail + missing + wrong: status, outcome, resp = "COMPLETED", "wrong", resp_ok
                    elif r < fail + missing + wrong + slow: status, outcome, resp, secs = "COMPLETED", "slow", resp_ok, random.uniform(60, 180)
                    elif r < fail + missing + wrong + slow + 0.05: status, outcome, resp = "COMPLETED", "clarify", resp_clarify
                    else: status, outcome, resp = "COMPLETED", "good", resp_ok
                    fb, comment = None, None
                    if outcome == "good" and random.random() < 0.15:
                        fb = "POSITIVE"; comment = random.choice(comments_pos) if random.random() < 0.4 else None
                    elif outcome in comments_neg and random.random() < 0.45:
                        fb = "NEGATIVE"; comment = random.choice(comments_neg[outcome]) if random.random() < 0.6 else None
                    msgs.append((str(uuid.uuid4()), conv, sp, email, ts, q, status, resp, round(secs, 1), fb, comment, "webapp"))
                    frustration = frustration * 0.9 + (0.0 if outcome == "good" else 1.0)
                    prev_q = q
                    if outcome == "good" and random.random() < 0.6: break
                if random.random() < 0.004 + 0.012 * frustration: break
            day += timedelta(days=1)
    msg_cols = ["message_id", "conversation_id", "space_id", "user_email", "created_at", "question", "status", "genie_response", "response_seconds", "feedback_rating", "feedback_comment", "source"]
    msg_df = pd.DataFrame(msgs, columns=msg_cols)
    ev = [(m[4], "aibiGenie", "createConversationMessage", m[3], m[2], m[1], m[0], None, "webapp") for m in msgs]
    ev += [(m[4] + timedelta(seconds=m[8] + 30), "aibiGenie", "updateConversationMessageFeedback", m[3], m[2], m[1], m[0], m[9], "webapp") for m in msgs if m[9]]
    ev_df = pd.DataFrame(ev, columns=["event_time", "service_name", "action_name", "user_email", "space_id", "conversation_id", "message_id", "feedback_rating", "source"])
    ev_df["event_date"] = pd.to_datetime(ev_df["event_time"]).dt.date
    return (pd.DataFrame(users, columns=["user_email", "department", "role", "provisioned_date"]),
            pd.DataFrame([(s[0], s[1], s[2]) for s in spaces], columns=["space_id", "space_name", "owning_department"]),
            msg_df, ev_df)

if DATA_MODE == "synthetic":
    users_df, spaces_df, msg_df, ev_df = generate_synthetic()
    for name, pdf in [("dim_genie_users", users_df), ("dim_genie_spaces", spaces_df), ("raw_genie_messages", msg_df), ("raw_genie_audit_events", ev_df)]:
        spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
        print(f"{name}: {len(pdf):,} rows")
    dbutils.notebook.exit("synthetic data written")

In [0]:
# Metadata layer: who asked, when, in which space, plus feedback ratings. Always available to admins.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
workspace_filter = ""
if WORKSPACE_SCOPE == "current":
    workspace_filter = f"AND workspace_id = {w.get_workspace_id()}"

GENIE_ACTIONS = [
    "createConversation", "createConversationMessage", "genieCreateConversationMessage",
    "genieStartConversationMessage", "createAgenticResponse", "genieCreateAgenticResponse",
    "regenerateConversationMessage", "updateConversationMessageFeedback", "getSpace",
]
actions_sql = ", ".join(f"'{a}'" for a in GENIE_ACTIONS)

spark.sql(f"""
  CREATE OR REPLACE TABLE raw_genie_audit_events AS
  SELECT
    event_time,
    event_date,
    workspace_id,
    service_name,
    action_name,
    user_identity.email                  AS user_email,
    request_params['space_id']           AS space_id,
    request_params['conversation_id']    AS conversation_id,
    request_params['message_id']         AS message_id,
    request_params['feedback_rating']    AS feedback_rating,
    COALESCE(request_params['source'], 'webapp') AS source,
    response.status_code                 AS status_code
  FROM system.access.audit
  WHERE event_date >= current_date() - INTERVAL {LOOKBACK_DAYS} DAYS
    {workspace_filter}
    AND (
      (service_name = 'aibiGenie' AND action_name IN ({actions_sql}))
      OR (service_name = 'genieChat' AND action_name = 'createGenieChatResponse')
    )
""")
display(spark.sql("SELECT action_name, count(*) events, count(DISTINCT user_email) users FROM raw_genie_audit_events GROUP BY 1 ORDER BY 2 DESC"))

In [0]:
# Content layer: question text, Genie's answer, status and timings. Optional (include_conversation_text).
import time
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor

def api_get(path, query=None, retries=5):
    for attempt in range(retries):
        try:
            return w.api_client.do("GET", path, query=query or {})
        except Exception as e:
            msg = str(e)
            if ("429" in msg or "RESOURCE_EXHAUSTED" in msg or "TEMPORARILY_UNAVAILABLE" in msg) and attempt < retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise

def paged(path, key, query=None, limit=None):
    query, out = dict(query or {}), []
    while True:
        resp = api_get(path, query)
        out.extend(resp.get(key, []) or [])
        token = resp.get("next_page_token")
        if not token or (limit and len(out) >= limit):
            return out[:limit] if limit else out
        query["page_token"] = token

# Spaces
spaces = paged("/api/2.0/genie/spaces", "spaces", {"page_size": 100})
space_rows = [(s.get("space_id") or s.get("id"), s.get("title") or s.get("display_name") or "", (s.get("description") or "")[:500]) for s in spaces]
print(f"Genie spaces visible to run-as identity: {len(space_rows)}")

cutoff_ms = int((datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)).timestamp() * 1000)

def ts(v):
    return datetime.fromtimestamp(int(v) / 1000, tz=timezone.utc).replace(tzinfo=None) if v else None

def fetch_conversation(space_id, conv):
    conv_id = conv.get("conversation_id") or conv.get("id")
    rows = []
    try:
        msgs = paged(f"/api/2.0/genie/spaces/{space_id}/conversations/{conv_id}/messages", "messages", {"page_size": 100})
    except Exception as e:
        print(f"skip {space_id}/{conv_id}: {str(e)[:120]}")
        return rows
    for m in msgs:
        texts, sql_desc = [], []
        for a in m.get("attachments") or []:
            if a.get("text"): texts.append(a["text"].get("content") or "")
            if a.get("query"): sql_desc.append(a["query"].get("description") or "")
        err = m.get("error") or {}
        created, updated = m.get("created_timestamp"), m.get("last_updated_timestamp")
        rows.append((
            m.get("message_id") or m.get("id"), conv_id, space_id, str(m.get("user_id") or ""),
            ts(created), (m.get("content") or "")[:4000], m.get("status"),
            (" ".join(texts + sql_desc) or err.get("error") or "")[:4000],
            err.get("type"),
            round((int(updated) - int(created)) / 1000.0, 1) if created and updated else None,
            (m.get("feedback") or {}).get("rating"),
        ))
    return rows

api_rows = []
if INCLUDE_TEXT:
    for sid, sname, _ in space_rows:
        try:
            convs = paged(f"/api/2.0/genie/spaces/{sid}/conversations", "conversations", {"include_all": "true", "page_size": 100}, limit=MAX_CONV)
        except Exception as e:
            print(f"Cannot list all conversations for '{sname}' ({sid}) - needs CAN MANAGE: {str(e)[:120]}")
            continue
        convs = [c for c in convs if int(c.get("last_updated_timestamp") or c.get("created_timestamp") or 0) >= cutoff_ms]
        with ThreadPoolExecutor(max_workers=8) as pool:
            for rows in pool.map(lambda c: fetch_conversation(sid, c), convs):
                api_rows.extend(rows)
        print(f"{sname}: {len(convs)} conversations")

api_schema = ("message_id string, conversation_id string, space_id string, api_user_id string, created_at timestamp, "
              "question string, status string, genie_response string, error_type string, response_seconds double, api_feedback_rating string")
spark.createDataFrame(api_rows, api_schema).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw_genie_api_messages")
spark.createDataFrame(space_rows, "space_id string, space_name string, space_description string").createOrReplaceTempView("api_spaces")
print(f"raw_genie_api_messages: {len(api_rows):,} rows")

In [0]:
# One row per user question. API rows carry text; audit-only rows (no API access) carry metadata only.
# User email comes from the audit log (conversation -> email), feedback from audit with API as fallback.
MESSAGE_ACTIONS = "('createConversationMessage','genieCreateConversationMessage','genieStartConversationMessage','createAgenticResponse','genieCreateAgenticResponse','createGenieChatResponse')"

spark.sql(f"""
CREATE OR REPLACE TABLE raw_genie_messages AS
WITH conv_owner AS (
  SELECT conversation_id, MIN_BY(user_email, event_time) AS user_email, MIN_BY(source, event_time) AS source
  FROM raw_genie_audit_events
  WHERE conversation_id IS NOT NULL AND user_email LIKE '%@%'
  GROUP BY 1
),
feedback AS (
  -- audit logs THUMBS_UP / THUMBS_DOWN; normalize to POSITIVE / NEGATIVE
  SELECT message_id,
         MAX_BY(CASE WHEN feedback_rating IN ('THUMBS_UP','POSITIVE') THEN 'POSITIVE'
                     WHEN feedback_rating IN ('THUMBS_DOWN','NEGATIVE') THEN 'NEGATIVE' END, event_time) AS feedback_rating
  FROM raw_genie_audit_events
  WHERE action_name = 'updateConversationMessageFeedback' AND message_id IS NOT NULL
  GROUP BY 1
),
api AS (
  SELECT a.message_id, a.conversation_id, a.space_id, o.user_email, a.created_at, a.question, a.status,
         a.genie_response, a.response_seconds,
         COALESCE(f.feedback_rating, CASE WHEN a.api_feedback_rating IN ('POSITIVE','THUMBS_UP') THEN 'POSITIVE'
                                          WHEN a.api_feedback_rating IN ('NEGATIVE','THUMBS_DOWN') THEN 'NEGATIVE' END) AS feedback_rating,
         CAST(NULL AS STRING) AS feedback_comment, COALESCE(o.source, 'webapp') AS source
  FROM raw_genie_api_messages a
  JOIN conv_owner o USING (conversation_id)
  LEFT JOIN feedback f USING (message_id)
),
audit_only AS (
  SELECT COALESCE(e.message_id, uuid()) AS message_id, e.conversation_id,
         -- Genie One chat (Slack/Teams/web chat) events have no space_id
         COALESCE(e.space_id, CASE WHEN e.service_name = 'genieChat' THEN 'genie_one_chat' ELSE 'unknown_space' END) AS space_id,
         e.user_email,
         e.event_time AS created_at, CAST(NULL AS STRING) AS question,
         CASE WHEN e.status_code >= 400 THEN 'FAILED' ELSE 'SUBMITTED' END AS status,
         CAST(NULL AS STRING) AS genie_response, CAST(NULL AS DOUBLE) AS response_seconds,
         f.feedback_rating, CAST(NULL AS STRING) AS feedback_comment, e.source
  FROM raw_genie_audit_events e
  LEFT JOIN feedback f ON f.message_id = e.message_id
  WHERE e.action_name IN {MESSAGE_ACTIONS}
    AND e.user_email LIKE '%@%'
    AND (e.conversation_id IS NULL OR e.conversation_id NOT IN (SELECT DISTINCT conversation_id FROM raw_genie_api_messages))
)
SELECT * FROM api
UNION ALL
SELECT * FROM audit_only
""")

# Spaces dimension (API names where visible, otherwise the space_id)
spark.sql("""
CREATE OR REPLACE TABLE dim_genie_spaces AS
SELECT m.space_id,
       COALESCE(MAX(s.space_name), CASE m.space_id WHEN 'genie_one_chat' THEN 'Genie One chat' ELSE m.space_id END) AS space_name,
       CAST(NULL AS STRING) AS owning_department
FROM (SELECT DISTINCT space_id FROM raw_genie_messages WHERE space_id IS NOT NULL) m
LEFT JOIN api_spaces s USING (space_id)
GROUP BY m.space_id
""")

# Users dimension (optionally enriched from a customer HR / identity table)
attr_join = f"LEFT JOIN {USER_ATTR_TABLE} a ON lower(a.user_email) = lower(u.user_email)" if USER_ATTR_TABLE else ""
dept = "COALESCE(a.department, 'Unknown')" if USER_ATTR_TABLE else "'Unknown'"
role = "COALESCE(a.role, 'Unknown')" if USER_ATTR_TABLE else "'Unknown'"
spark.sql(f"""
CREATE OR REPLACE TABLE dim_genie_users AS
SELECT u.user_email, {dept} AS department, {role} AS role, u.provisioned_date
FROM (SELECT user_email, DATE(MIN(created_at)) AS provisioned_date FROM raw_genie_messages GROUP BY 1) u
{attr_join}
""")

for t in ["raw_genie_audit_events", "raw_genie_api_messages", "raw_genie_messages", "dim_genie_spaces", "dim_genie_users"]:
    print(t, spark.table(t).count())